# TP2 – Genetic Algorithm (Python version)
**Goal:** maximise f(x) = -x² + 4x on the interval [1, 30].

**Fixed by the assignment**
- Population of 4 individuals (chromosomes), initial population: 18, 6, 11, 27
- Each individual is encoded on 5 bits (genes)
- Uniform crossover
- 4 scenarios to test:

| Scenario | Pc | Pm | max-generation |
|---|---|---|---|
| 1 | 0.75 | 0.005 | 30 |
| 2 | 0.75 | 0.005 | 50 |
| 3 | 0.90 | 0.01 | 30 |
| 4 | 0.90 | 0.01 | 50 |

**Statistics collected for each scenario:** execution time, fitness vector (one value per iteration), average fitness, maximum fitness, number of crossovers, number of mutations, number of accepted new solutions.

## 0. Imports

In [2]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Initialization and fitness function

**Objective function.** We maximise f(x) = -x² + 4x on the interval [1, 30].

**Chromosome representation.** Each solution x is encoded as a chromosome of 5 bits
(genes), because 2⁵ = 32 values can be represented (0 to 31) and the interval needs
values up to 30. Example: 18 → 10010, 6 → 00110, 11 → 01011, 27 → 11011.

**Initial population.** 4 individuals given by the assignment: 18, 6, 11, 27.

**Fitness evaluation.** The fitness of a chromosome is f(x), where x is its decoded
integer value. All four initial individuals have a negative fitness, because the
maximum of f is at x = 2 and they are far from it.

**Handling x = 0 and x = 31.** 5 bits can also represent 0 and 31, which are outside
[1, 30]. Crossover or mutation can produce them, so we use a penalty: any chromosome
outside [1, 30] gets a very bad fitness (-10⁹). It is therefore never selected and
never accepted into the population.

In [3]:
N_BITS = 5  # chromosome length
X_MIN, X_MAX = 1, 30  # valid interval for x
POP_SIZE = 4  # population size
INITIAL_POP = [18, 6, 11, 27]  # given initial population
PENALTY = -1e9  # fitness of invalid chromosomes (x = 0 or x = 31)


def encode(x):
    """integer -> list of 5 bits, e.g. 18 -> [1,0,0,1,0]"""
    return [int(b) for b in format(x, f"0{N_BITS}b")]


def decode(chrom):
    """list of 5 bits -> integer, e.g. [1,0,0,1,0] -> 18"""
    return int("".join(map(str, chrom)), 2)


def f(x):
    """objective function"""
    return -(x**2) + 4 * x


def fitness(chrom):
    """fitness of a chromosome, with a penalty outside [1, 30]"""
    x = decode(chrom)
    if x < X_MIN or x > X_MAX:
        return PENALTY
    return f(x)


def initial_population():
    return [encode(x) for x in INITIAL_POP]


# quick check
for x in INITIAL_POP:
    print(f"x = {x:2d}  chromosome = {encode(x)}  fitness = {fitness(encode(x))}")
print("x = 0 ->", fitness(encode(0)), "| x = 31 ->", fitness(encode(31)))


x = 18  chromosome = [1, 0, 0, 1, 0]  fitness = -252
x =  6  chromosome = [0, 0, 1, 1, 0]  fitness = -12
x = 11  chromosome = [0, 1, 0, 1, 1]  fitness = -77
x = 27  chromosome = [1, 1, 0, 1, 1]  fitness = -621
x = 0 -> -1000000000.0 | x = 31 -> -1000000000.0
